In [11]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)

spark = (
    SparkSession.builder
    .appName("Lesson21-DataFrames-SparkSQL")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

In [12]:
customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("balance", DoubleType(), True),
    StructField("vip_status", StringType(), True)
])

In [15]:
df_customers = (
    spark.read
    .option("header", "true")
    .schema(customer_schema)
    .csv("s3a://matrix/raw/customers_new.csv")
)

In [16]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- balance: double (nullable = true)
 |-- vip_status: string (nullable = true)



In [17]:
card_trn_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("s3a://matrix/raw/card_trn_new.csv")
)

In [18]:
card_trn_raw.printSchema()

root
 |-- trn_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- trn_date: date (nullable = true)
 |-- merchant: string (nullable = true)



In [19]:
trn_items_raw = spark.read.json("s3a://matrix/raw/trn_items.json")

In [20]:
trn_items_raw.printSchema()

root
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product_name: string (nullable = true)
 |    |    |-- qty: long (nullable = true)
 |    |    |-- unit_price: double (nullable = true)
 |-- trn_id: string (nullable = true)



In [22]:
customers_clean = (
    df_customers
    .withColumn("balance", F.when(F.col("balance").isNull(), 0.0).otherwise(F.col("balance")))
    .withColumn(
        "age_group", F.when(F.col("age") < 25, "young")
         .when((F.col("age") >= 25) & (F.col("age") <= 40), "adult")
         .otherwise("senior"))
    .withColumnRenamed("vip_status", "flg_is_vip")
    .withColumn("insert_date", F.current_date())
)


In [27]:
customer_trn = customers_clean.join(card_trn_raw,on="customer_id")

In [25]:
customer_trn.show()

+-----------+---------------+---+-------+----------+---------+-----------+------+------+----------+------------+
|customer_id|  customer_name|age|balance|flg_is_vip|age_group|insert_date|trn_id|amount|  trn_date|    merchant|
+-----------+---------------+---+-------+----------+---------+-----------+------+------+----------+------------+
|       C001|  Rashad Aliyev| 23|1230.15|         Y|    young| 2026-09-15| T0001| 67.14|2026-03-09|       SOCAR|
|       C001|  Rashad Aliyev| 23|1230.15|         Y|    young| 2026-09-15| T0002|265.14|2026-07-11|     Ecoteks|
|       C002|Sabina Mammadov| 60|3651.47|         Y|   senior| 2026-09-15| T0003|448.81|2026-02-26|        Wolt|
|       C002|Sabina Mammadov| 60|3651.47|         Y|   senior| 2026-09-15| T0004|224.61|2026-05-11|       Bravo|
|       C002|Sabina Mammadov| 60|3651.47|         Y|   senior| 2026-09-15| T0005| 53.79|2026-08-09|        Wolt|
|       C003|Sevinj Karimova| 19|    0.0|         Y|    young| 2026-09-15| T0006| 270.4|2026-04-

In [28]:
customer_wthout_trn = customers_clean.join(card_trn_raw,on="customer_id", how="left_anti")

In [30]:
from pyspark.sql.functions import broadcast


In [31]:
customer_trn_broadcast = customers_clean.join(
    broadcast(card_trn_raw),
    on="customer_id",
    how="inner"
)

In [32]:
customer_trn.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#0, customer_name#1, age#2, balance#56, flg_is_vip#69, age_group#62, 2026-09-15 AS insert_date#76, trn_id#27, amount#29, trn_date#30, merchant#31]
   +- BroadcastHashJoin [customer_id#0], [customer_id#28], Inner, BuildLeft, false
      :- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=128]
      :  +- Project [customer_id#0, customer_name#1, age#2, CASE WHEN isnull(balance#3) THEN 0.0 ELSE balance#3 END AS balance#56, vip_status#4 AS flg_is_vip#69, CASE WHEN (age#2 < 25) THEN young WHEN ((age#2 >= 25) AND (age#2 <= 40)) THEN adult ELSE senior END AS age_group#62]
      :     +- Filter isnotnull(customer_id#0)
      :        +- FileScan csv [customer_id#0,customer_name#1,age#2,balance#3,vip_status#4] Batched: false, DataFilters: [isnotnull(customer_id#0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[s3a://matrix/raw/customers_new.csv], PartitionFilters: [], P

In [33]:
customer_trn_broadcast.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#0, customer_name#1, age#2, balance#56, flg_is_vip#69, age_group#62, 2026-09-15 AS insert_date#76, trn_id#27, amount#29, trn_date#30, merchant#31]
   +- BroadcastHashJoin [customer_id#0], [customer_id#28], Inner, BuildRight, false
      :- Project [customer_id#0, customer_name#1, age#2, CASE WHEN isnull(balance#3) THEN 0.0 ELSE balance#3 END AS balance#56, vip_status#4 AS flg_is_vip#69, CASE WHEN (age#2 < 25) THEN young WHEN ((age#2 >= 25) AND (age#2 <= 40)) THEN adult ELSE senior END AS age_group#62]
      :  +- Filter isnotnull(customer_id#0)
      :     +- FileScan csv [customer_id#0,customer_name#1,age#2,balance#3,vip_status#4] Batched: false, DataFilters: [isnotnull(customer_id#0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[s3a://matrix/raw/customers_new.csv], PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<customer_id:string,customer_name:string,age:int,balance

In [37]:
customer_trn.createOrReplaceTempView("customer_trn")


top2_transactions = spark.sql("""
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY amount DESC
            ) AS rn
        FROM customer_trn
    )
    WHERE rn <= 2
""")

In [40]:
top2_transactions.show(truncate = False)

+-----------+---------------+---+-------+----------+---------+-----------+------+------+----------+------------+---+
|customer_id|customer_name  |age|balance|flg_is_vip|age_group|insert_date|trn_id|amount|trn_date  |merchant    |rn |
+-----------+---------------+---+-------+----------+---------+-----------+------+------+----------+------------+---+
|C001       |Rashad Aliyev  |23 |1230.15|Y         |young    |2026-09-15 |T0002 |265.14|2026-07-11|Ecoteks     |1  |
|C001       |Rashad Aliyev  |23 |1230.15|Y         |young    |2026-09-15 |T0001 |67.14 |2026-03-09|SOCAR       |2  |
|C002       |Sabina Mammadov|60 |3651.47|Y         |senior   |2026-09-15 |T0003 |448.81|2026-02-26|Wolt        |1  |
|C002       |Sabina Mammadov|60 |3651.47|Y         |senior   |2026-09-15 |T0004 |224.61|2026-05-11|Bravo       |2  |
|C003       |Sevinj Karimova|19 |0.0    |Y         |young    |2026-09-15 |T0009 |325.72|2026-06-24|SOCAR       |1  |
|C003       |Sevinj Karimova|19 |0.0    |Y         |young    |20

In [41]:

agg_transactions = spark.sql("""
        SELECT customer_id,customer_name,age_group,flg_is_vip,
        COUNT(*) AS transaction_count,
        SUM(amount) AS total_amount,
        AVG(amount) AS avg_amount,
        MAX(amount) AS max_amount
        FROM customer_trn
        GROUP BY
        customer_id,
        customer_name,
        age_group,
        flg_is_vip
    ORDER BY total_amount DESC
""")

In [42]:
agg_transactions.show(truncate = False)

+-----------+-----------------+---------+----------+-----------------+------------------+------------------+----------+
|customer_id|customer_name    |age_group|flg_is_vip|transaction_count|total_amount      |avg_amount        |max_amount|
+-----------+-----------------+---------+----------+-----------------+------------------+------------------+----------+
|C003       |Sevinj Karimova  |young    |Y         |6                |1471.8700000000001|245.3116666666667 |325.72    |
|C016       |Gunel Mammadov   |senior   |Y         |4                |1471.15           |367.7875          |417.23    |
|C013       |Orkhan Mammadov  |adult    |Y         |6                |1445.7            |240.95000000000002|421.3     |
|C005       |Vusal Guliyev    |young    |N         |5                |1117.71           |223.542           |429.31    |
|C008       |Aysel Mammadov   |young    |N         |5                |1106.85           |221.36999999999998|418.37    |
|C002       |Sabina Mammadov  |senior   

In [44]:
card_trn_raw.createOrReplaceTempView("card_trn")
trn_items_raw.createOrReplaceTempView("trn_items")

In [45]:
joined_items = spark.sql("""
    SELECT
        t.trn_id,
        t.amount,
        i.items
    FROM card_trn t
    INNER JOIN trn_items i
        ON t.trn_id = i.trn_id
""")

In [47]:
joined_items.createOrReplaceTempView("joined_items")

In [48]:
items_exploded = spark.sql("""
    SELECT
        trn_id,
        amount,
        EXPLODE(items) AS item
    FROM joined_items
""")

In [49]:
items_exploded.show(truncate = False)

+------+------+-------------------+
|trn_id|amount|item               |
+------+------+-------------------+
|T0001 |67.14 |{Cheese, 1, 5.5}   |
|T0001 |67.14 |{Charger, 3, 12.5} |
|T0001 |67.14 |{Bread, 1, 1.2}    |
|T0002 |265.14|{Snack Bar, 1, 1.5}|
|T0002 |265.14|{Bread, 1, 1.2}    |
|T0002 |265.14|{Water, 1, 0.9}    |
|T0003 |448.81|{Charger, 3, 12.5} |
|T0003 |448.81|{Milk, 2, 2.1}     |
|T0003 |448.81|{Coffee, 1, 8.9}   |
|T0003 |448.81|{Detergent, 3, 6.4}|
|T0004 |224.61|{Coffee, 3, 8.9}   |
|T0005 |53.79 |{Snack Bar, 3, 1.5}|
|T0006 |270.4 |{Bread, 1, 1.2}    |
|T0006 |270.4 |{Coffee, 2, 8.9}   |
|T0006 |270.4 |{Shampoo, 2, 4.7}  |
|T0006 |270.4 |{Water, 1, 0.9}    |
|T0007 |270.17|{Snack Bar, 2, 1.5}|
|T0008 |116.88|{Milk, 3, 2.1}     |
|T0008 |116.88|{Snack Bar, 3, 1.5}|
|T0009 |325.72|{Pen, 3, 0.8}      |
+------+------+-------------------+
only showing top 20 rows



In [50]:
items_exploded.printSchema()

root
 |-- trn_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- item: struct (nullable = true)
 |    |-- product_name: string (nullable = true)
 |    |-- qty: long (nullable = true)
 |    |-- unit_price: double (nullable = true)



In [51]:
items_exploded.createOrReplaceTempView("items_exploded")

In [52]:
items_with_total = spark.sql("""
    SELECT
        trn_id,
        amount,
        item.qty,
        item.unit_price,
        item.qty * item.unit_price AS line_total
    FROM items_exploded
""")

In [53]:
items_with_total.show(truncate=False)

+------+------+---+----------+------------------+
|trn_id|amount|qty|unit_price|line_total        |
+------+------+---+----------+------------------+
|T0001 |67.14 |1  |5.5       |5.5               |
|T0001 |67.14 |3  |12.5      |37.5              |
|T0001 |67.14 |1  |1.2       |1.2               |
|T0002 |265.14|1  |1.5       |1.5               |
|T0002 |265.14|1  |1.2       |1.2               |
|T0002 |265.14|1  |0.9       |0.9               |
|T0003 |448.81|3  |12.5      |37.5              |
|T0003 |448.81|2  |2.1       |4.2               |
|T0003 |448.81|1  |8.9       |8.9               |
|T0003 |448.81|3  |6.4       |19.200000000000003|
|T0004 |224.61|3  |8.9       |26.700000000000003|
|T0005 |53.79 |3  |1.5       |4.5               |
|T0006 |270.4 |1  |1.2       |1.2               |
|T0006 |270.4 |2  |8.9       |17.8              |
|T0006 |270.4 |2  |4.7       |9.4               |
|T0006 |270.4 |1  |0.9       |0.9               |
|T0007 |270.17|2  |1.5       |3.0               |


In [54]:
items_with_total.createOrReplaceTempView("items_total")

In [55]:
transaction_comparison = spark.sql("""
    SELECT
        trn_id,
        amount,
        SUM(line_total) AS items_total,
        SUM(line_total) - amount AS difference
    FROM items_total
    GROUP BY
        trn_id,
        amount
""")

In [56]:
transaction_comparison.show(truncate =False)

+------+------+------------------+-------------------+
|trn_id|amount|items_total       |difference         |
+------+------+------------------+-------------------+
|T0028 |319.99|27.000000000000004|-292.99            |
|T0001 |67.14 |44.2              |-22.939999999999998|
|T0021 |60.96 |25.0              |-35.96             |
|T0003 |448.81|69.80000000000001 |-379.01            |
|T0043 |417.23|18.3              |-398.93            |
|T0038 |89.64 |55.6              |-34.04             |
|T0027 |251.5 |8.0               |-243.5             |
|T0044 |221.11|19.6              |-201.51000000000002|
|T0012 |394.75|57.7              |-337.05            |
|T0030 |233.8 |54.00000000000001 |-179.8             |
|T0016 |429.31|50.900000000000006|-378.40999999999997|
|T0017 |149.25|18.700000000000003|-130.55            |
|T0024 |143.71|27.7              |-116.01            |
|T0026 |104.99|24.200000000000003|-80.78999999999999 |
|T0005 |53.79 |4.5               |-49.29             |
|T0018 |16

In [57]:
card_trn_clean = (
    card_trn_raw
    .withColumn("trn_month", F.month("trn_date"))
)

In [58]:
customer_monthly = (
    card_trn_clean
    .groupBy("customer_id")
    .pivot("trn_month")
    .sum("amount")
    .na.fill(0)
)

In [60]:
customer_monthly.show(truncate =False)

+-----------+-----------------+------+------+-----------------+------+------+------------------+------+
|customer_id|1                |2     |3     |4                |5     |6     |7                 |8     |
+-----------+-----------------+------+------+-----------------+------+------+------------------+------+
|C006       |0.0              |0.0   |0.0   |0.0              |0.0   |0.0   |0.0               |166.53|
|C010       |0.0              |0.0   |0.0   |0.0              |251.5 |0.0   |0.0               |0.0   |
|C007       |442.89           |0.0   |0.0   |0.0              |0.0   |0.0   |0.0               |0.0   |
|C012       |641.71           |0.0   |0.0   |0.0              |0.0   |0.0   |0.0               |0.0   |
|C003       |0.0              |0.0   |198.47|540.5699999999999|116.88|325.72|290.23            |0.0   |
|C015       |0.0              |282.38|0.0   |0.0              |0.0   |0.0   |0.0               |0.0   |
|C004       |0.0              |0.0   |0.0   |0.0              |0

In [62]:
customer_summary = (
    customer_trn
    .groupBy(
        "customer_id",
        "customer_name",
        "age_group",
        "flg_is_vip"
    )
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("avg_amount"),
        F.max("amount").alias("max_amount")
    )
)

In [63]:
customer_summary.write \
    .format("parquet") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("s3a://matrix/silver/customer_summary")

In [64]:
customer_summary.show(truncate=False)

+-----------+-----------------+---------+----------+-----------------+------------------+------------------+----------+
|customer_id|customer_name    |age_group|flg_is_vip|transaction_count|total_amount      |avg_amount        |max_amount|
+-----------+-----------------+---------+----------+-----------------+------------------+------------------+----------+
|C014       |Nargiz Ismayilova|senior   |N         |1                |329.06            |329.06            |329.06    |
|C012       |Sevinj Guliyev   |young    |Y         |2                |641.71            |320.855           |407.91    |
|C009       |Kamran Nuriyev   |adult    |Y         |1                |104.99            |104.99            |104.99    |
|C010       |Zeynab Abbasova  |young    |N         |1                |251.5             |251.5             |251.5     |
|C003       |Sevinj Karimova  |young    |Y         |6                |1471.8700000000001|245.3116666666667 |325.72    |
|C011       |Nigar Abbasova   |adult    

In [65]:
cust_check = spark.read.parquet(
    "s3a://matrix/silver/customer_summary"
)

In [67]:
cust_check.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- flg_is_vip: string (nullable = true)
 |-- transaction_count: long (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- avg_amount: double (nullable = true)
 |-- max_amount: double (nullable = true)

